# Notebook 01 — Exploração: Paralelismo e Memória em Inferência de LLMs

## Objetivos

1. Explorar o workload sintético de requisições de inferência LLM
2. Visualizar distribuições de comprimento de prompt e geração
3. Analisar requisitos de memória KV-Cache para diferentes modelos
4. Compreender a Lei de Amdahl vs. Gustafson para limites de paralelismo
5. Comparar alocação contígua vs. PagedAttention (Kwon et al., 2023)

## Contexto Teórico

> *"A demanda computacional de modelos de linguagem de grande escala transformou
> fundamentalmente a arquitetura de sistemas de inferência. Enquanto o GPT-2 (2019)
> operava confortavelmente em uma única GPU com 1.5 bilhões de parâmetros, modelos
> contemporâneos como LLaMA-2 70B e GPT-4 exigem dezenas ou centenas de aceleradores
> operando em conjunto."* — Material da Aula 07

**Referências:**
- Brown et al. (2020). Language Models are Few-Shot Learners. NeurIPS.
- Kwon et al. (2023). PagedAttention. SOSP 2023.
- Gustafson (1988). Reevaluating Amdahl's Law. CACM.

In [ ]:
# Imports
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adiciona diretório raiz ao path para importar src/
sys.path.insert(0, os.path.abspath('..'))

from src.data_preprocessing import WorkloadPreprocessor
from src.model import amdahl_speedup, gustafson_speedup, PagedKVCacheManager
from src.evaluation import ServingMetricsEvaluator
from src.utils import compute_memory_savings

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print('Imports carregados com sucesso.')

## 1. Carregamento e Exploração do Workload

O dataset simula requisições de inferência para LLMs com distribuições realistas,
calibradas com base nos padrões do dataset ShareGPT reportados por **Kwon et al. (2023)**
na seção *PagedAttention e Gerenciamento de KV-Cache* do material da aula.

In [ ]:
# Carrega workload sintético
preprocessor = WorkloadPreprocessor(seed=42)
df = preprocessor.load_data('../data/raw/dataset.csv')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
print(f'\nModelos simulados:')
print(df['model_size_b'].value_counts().sort_index())
df.head(10)

In [ ]:
# Estatísticas descritivas
print('Estatísticas do Workload:')
df[['prompt_tokens', 'max_gen_tokens', 'actual_gen_tokens']].describe().round(1)

## 2. Distribuições de Comprimento de Sequência

Conforme descrito na seção *Continuous Batching e Scheduling Dinâmico* (Yu et al., 2022):

> *"Requisições curtas subutilizam buffers grandes, enquanto requisições longas podem
> falhar por falta de memória pré-alocada."*

A variabilidade de comprimento é o principal motivador para continuous batching e PagedAttention.

In [ ]:
# Distribuições de tokens (prompt e geração)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Prompt tokens
axes[0].hist(df['prompt_tokens'], bins=50, alpha=0.7, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Prompt Tokens')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Prompt Tokens\n(Lognormal, padrão ShareGPT)')
axes[0].axvline(df['prompt_tokens'].median(), color='red', linestyle='--',
                label=f"Mediana={df['prompt_tokens'].median():.0f}")
axes[0].legend()

# Generated tokens
axes[1].hist(df['actual_gen_tokens'], bins=50, alpha=0.7, color='coral', edgecolor='white')
axes[1].set_xlabel('Tokens Gerados')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição de Tokens Gerados')
axes[1].axvline(df['actual_gen_tokens'].median(), color='red', linestyle='--',
                label=f"Mediana={df['actual_gen_tokens'].median():.0f}")
axes[1].legend()

# Total tokens (prompt + geração)
total = df['prompt_tokens'] + df['actual_gen_tokens']
axes[2].hist(total, bins=50, alpha=0.7, color='seagreen', edgecolor='white')
axes[2].set_xlabel('Total Tokens (prompt + geração)')
axes[2].set_ylabel('Frequência')
axes[2].set_title('Distribuição de Tokens Totais')
axes[2].axvline(total.median(), color='red', linestyle='--',
                label=f'Mediana={total.median():.0f}')
axes[2].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_token_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Variabilidade de comprimento: min={total.min()}, max={total.max()}, '
      f'ratio={total.max()/total.min():.1f}x')

## 3. Análise de Padrões de Chegada

Os intervalos entre chegadas seguem um processo de Poisson conforme
o tráfego modelado por **Yu et al. (2022)** para sistemas Orca.
Compreender o padrão de chegada é essencial para o design do scheduler.

In [ ]:
# Padrões de chegada
inter_arrivals = df['arrival_time_ms'].diff().dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Intervalo entre chegadas
axes[0].hist(inter_arrivals, bins=50, alpha=0.7, color='steelblue', edgecolor='white',
             density=True)
axes[0].set_xlabel('Intervalo entre Chegadas (ms)')
axes[0].set_ylabel('Densidade')
axes[0].set_title('Distribuição de Intervalos\n(Processo de Poisson)')

# Taxa de chegada ao longo do tempo
window_ms = 1000  # 1 segundo
bins = np.arange(0, df['arrival_time_ms'].max() + window_ms, window_ms)
counts, _ = np.histogram(df['arrival_time_ms'], bins=bins)
axes[1].plot(bins[:-1] / 1000, counts, color='steelblue', linewidth=1)
axes[1].axhline(counts.mean(), color='red', linestyle='--',
                label=f'Média={counts.mean():.1f} req/s')
axes[1].set_xlabel('Tempo (s)')
axes[1].set_ylabel('Requisições / segundo')
axes[1].set_title('Taxa de Chegada ao Longo do Tempo')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/01_arrival_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Análise de KV-Cache por Modelo

Conforme seção *PagedAttention e Gerenciamento de KV-Cache* do material da aula:

> *"Para uma sequência com $n$ tokens e blocos de tamanho $B$, o número de blocos
> necessários é $\lceil n/B \rceil$."* — Kwon et al. (2023)

O tamanho do KV-Cache cresce linearmente com o comprimento de sequência,
número de camadas e dimensão oculta do modelo.

In [ ]:
# Calcula memória KV-Cache usando PagedAttention
df_kv = preprocessor.compute_kv_cache_memory(df, block_size=16, dtype_bytes=2)

print('KV-Cache por modelo:')
kv_summary = df_kv.groupby('model_size_b').agg({
    'total_tokens': ['mean', 'max'],
    'num_blocks': ['mean', 'max'],
    'total_kv_cache_mb': ['mean', 'median', 'max'],
}).round(2)
kv_summary

In [ ]:
# Visualização de KV-Cache por modelo
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KV-Cache distribution por modelo
for model_size in sorted(df_kv['model_size_b'].unique()):
    subset = df_kv[df_kv['model_size_b'] == model_size]
    axes[0].hist(subset['total_kv_cache_mb'], bins=40, alpha=0.5,
                 label=f'{int(model_size)}B')

axes[0].set_xlabel('KV-Cache (MB)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de KV-Cache por Modelo')
axes[0].legend()

# Blocos PagedAttention necessários
sns.boxplot(data=df_kv, x='model_size_b', y='num_blocks', ax=axes[1],
            palette='viridis')
axes[1].set_xlabel('Modelo (bilhões de parâmetros)')
axes[1].set_ylabel('Blocos PagedAttention (B=16)')
axes[1].set_title('Blocos PagedAttention Necessários\nceil(total_tokens / 16)')

plt.tight_layout()
plt.savefig('../outputs/figures/01_kv_cache_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. PagedAttention vs. Alocação Contígua

Conforme material da aula (seção *PagedAttention e Gerenciamento de KV-Cache*):

> *"PagedAttention elimina fragmentação externa e interna, aumentando a utilização
> de memória de aproximadamente 50% para mais de 95% em workloads reais."*

Vamos validar empiricamente esta afirmação com nosso workload.

In [ ]:
# Análise de economia de memória: contígua vs. PagedAttention
total_tokens = (df['prompt_tokens'] + df['actual_gen_tokens']).values

evaluator = ServingMetricsEvaluator(output_dir='../outputs/figures')
fig = evaluator.plot_kv_cache_comparison(
    total_tokens,
    block_size=16,
    save_name='01_paged_vs_contiguous.png',
)
plt.show()

# Métricas numéricas
savings = compute_memory_savings(total_tokens, block_size=16)
print(f"\nResultados (Kwon et al., 2023):")
print(f"  Utilização contígua:      {savings['contiguous_utilization']:.1%}")
print(f"  Utilização PagedAttention: {savings['paged_utilization']:.1%}")
print(f"  Economia vs. contígua:    {savings['savings_vs_contiguous']:.1%}")

## 6. Lei de Amdahl vs. Gustafson

Conforme seção *Fundamentos de Paralelismo em Deep Learning* do material:

> *"A Lei de Amdahl limita os ganhos de paralelização ao estabelecer que o speedup
> máximo é determinado pela fração sequencial do código. Gustafson (1988) propôs
> uma perspectiva complementar: em workloads escaláveis, os ganhos podem ser maiores."*

Isto é relevante para inferência de LLMs, onde aumentar o batch size pode
proporcionar ganhos próximos ao linear.

In [ ]:
# Amdahl vs. Gustafson
fig = evaluator.plot_amdahl_gustafson(
    serial_fractions=[0.01, 0.05, 0.10, 0.25],
    max_processors=64,
    save_name='01_amdahl_gustafson.png',
)
plt.show()

# Exemplos numéricos
print('\nExemplos para s=10% (fração serial):')
for p in [2, 4, 8, 16, 32, 64]:
    a = amdahl_speedup(0.10, p)
    g = gustafson_speedup(0.10, p)
    print(f'  P={p:2d} processadores: Amdahl={a:.2f}x, Gustafson={g:.2f}x')

## 7. Exploração Interativa do PagedKVCacheManager

Vamos simular a alocação e liberação de blocos conforme descrito na seção
*PagedAttention e Gerenciamento de KV-Cache*:

> *"A tabela de blocos $T$ mapeia índices de blocos lógicos $i$ para endereços
> físicos: $T[i] \rightarrow \text{addr}_i$."*

In [ ]:
# Demonstração de PagedKVCacheManager
kv_manager = PagedKVCacheManager(num_blocks=64, block_size=16)

print('Estado inicial:')
print(f'  Blocos livres: {kv_manager.free_blocks}/{kv_manager.num_blocks}')

# Aloca 5 sequências com comprimentos variados
sequences = [(0, 100), (1, 256), (2, 50), (3, 180), (4, 30)]
for seq_id, n_tokens in sequences:
    success = kv_manager.allocate(seq_id, n_tokens)
    blocks_used = len(kv_manager.block_tables.get(seq_id, []))
    print(f'  Seq {seq_id} ({n_tokens} tokens): alocou {blocks_used} blocos, '
          f'sucesso={success}')

stats = kv_manager.get_memory_stats()
print(f'\nApós alocações:')
print(f'  Blocos usados: {stats["used_blocks"]}/{stats["total_blocks"]} '
      f'({stats["utilization"]:.1%})')

# Libera sequências completadas
for seq_id in [0, 2, 4]:
    freed = kv_manager.free(seq_id)
    print(f'  Liberou seq {seq_id}: {freed} blocos')

stats = kv_manager.get_memory_stats()
print(f'\nApós liberações:')
print(f'  Blocos usados: {stats["used_blocks"]}/{stats["total_blocks"]} '
      f'({stats["utilization"]:.1%})')
print(f'  Blocos livres: {stats["free_blocks"]}')

## 8. Prefix Sharing

Conforme material da aula:

> *"Requisições que compartilham o mesmo prefixo (como um system prompt) podem
> referenciar os mesmos blocos físicos de KV-Cache, multiplicando a eficiência
> de memória para chatbots onde milhares de conversas compartilham instruções
> iniciais idênticas."*

In [ ]:
# Demonstração de Prefix Sharing
kv_shared = PagedKVCacheManager(num_blocks=128, block_size=16)

# System prompt compartilhado (256 tokens = 16 blocos)
system_prompt_tokens = 256
kv_shared.allocate(seq_id=100, num_tokens=system_prompt_tokens)
prefix_blocks = len(kv_shared.block_tables[100])
print(f'System prompt: {system_prompt_tokens} tokens -> {prefix_blocks} blocos')

# 5 conversas compartilham o system prompt
n_conversations = 5
for i in range(n_conversations):
    conv_id = 200 + i
    user_tokens = np.random.randint(20, 100)  # tokens únicos por conversa
    # Compartilha blocos do system prompt
    kv_shared.share_prefix(source_seq_id=100, target_seq_id=conv_id,
                           num_prefix_blocks=prefix_blocks)
    # Aloca blocos adicionais para conteúdo único
    kv_shared.extend(conv_id, user_tokens)

stats = kv_shared.get_memory_stats()
print(f'\nApós prefix sharing com {n_conversations} conversas:')
print(f'  Blocos usados: {stats["used_blocks"]}')
print(f'  Blocos compartilhados: {stats["shared_blocks"]}')
print(f'  Economia: blocos de system prompt alocados 1× em vez de {n_conversations+1}×')
saving = prefix_blocks * n_conversations
print(f'  Blocos economizados: {saving} ({saving * 16} tokens de capacidade)')

## Resumo

Neste notebook exploramos:

1. **Workload de inferência LLM** — distribuições lognormais de comprimento (Kwon et al., 2023)
2. **Memória KV-Cache** — cresce linearmente com sequência × camadas × hidden_dim
3. **PagedAttention vs. contígua** — utilização >95% vs. ~50% (validação empírica)
4. **Amdahl vs. Gustafson** — limites de paralelismo para workloads fixos vs. escaláveis
5. **Prefix sharing** — economia de memória para chatbots com system prompt compartilhado

**→ Próximo notebook:** [02_treinamento.ipynb](02_treinamento.ipynb) — Simulação de
serving distribuído com continuous batching e tensor/pipeline parallelism.